In [1]:
import os, sys; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import copy, numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART2
from src.target import load_claims
from src.evaluate import evaluate

def D2(act, pv, w):
    return evaluate(pd.Series(np.asarray(act)), np.asarray(pv), pd.Series(np.asarray(w)), show=False)["D2"]

ours = copy.deepcopy(PART2)
T.AMOUNTS_NET, T.ROW_COMPLETENESS = bool(ours["amounts_net"]), bool(ours["row_completeness"])
claims = load_claims(ours["data_dir"])
parts = []
for line in ours["lines"]:
    d = P.build_line(line, ours, claims)
    tr0 = d["tr"].merge(d["p"][["row_id", "policy_id"]], on="row_id")
    vy, tys = ours["folds"][0]
    tr, va = tr0[tr0.year.isin(tys)], tr0[tr0.year == vy]
    rate, _ = P.predict_rate(line, ours, tr, va, d["feats"])
    rate = np.asarray(rate, float) + d.get("sep_rate", 0.0)
    act = (va.target_all if "target_all" in va else va.target).to_numpy()
    parts.append(pd.DataFrame({"line": line, "row_id": va.row_id.to_numpy(), "policy_id": va.policy_id.astype(str).to_numpy(),
                               "new_business": va.new_business.astype(bool).to_numpy(),
                               "w": va.weight.to_numpy(), "rate": rate, "act": act}))
V = pd.concat(parts, ignore_index=True)
V["pv"] = V.rate * V.w
d2 = D2(V.act, V.pv, V.w)
print(f"pooled D2 {d2:.4f} (p2b 0.1077) | rows {len(V):,}")
assert abs(d2 - 0.1077) < 0.0005, "path differs from p2b"
Path("results").mkdir(exist_ok=True)
V.to_csv("results/valid2024_rows_ours.csv", index=False)
print("saved results/valid2024_rows_ours.csv")

20:17:37 motor: developed training table 979,968 rows
20:18:02 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
20:18:02 buildings: fire priced separately at 97.05 per exposure (developed)
20:18:02 buildings: own GLM for ['flood']; main GLM on the other perils
20:18:07 buildings: developed training table 707,726 rows
20:18:15 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
20:18:19 contents: developed training table 721,617 rows
pooled D2 0.1077 (p2b 0.1077) | rows 59,202
saved results/valid2024_rows_ours.csv


In [2]:
# (a) credibility by line x new business
g = (V.assign(loss_row=V.act > 0)
       .groupby(["line", "new_business"])
       .agg(rows=("w", "size"), exposure=("w", "sum"), loss_rows=("loss_row", "sum"), A=("act", "sum"), E=("pv", "sum")))
g["A/E"] = g.A / g.E
print(g.drop(columns=["A", "E"]).round(3).to_string(), "\n")

# (b) slope alpha by line, total held fixed
def tilt(df, a):
    rb = np.average(df.rate, weights=df.w)
    r = rb * (df.rate / rb) ** a
    return r * (df.rate * df.w).sum() / (r * df.w).sum() * df.w

alphas = np.round(np.arange(0.70, 1.61, 0.05), 2)
half = (pd.util.hash_pandas_object(V.policy_id, index=False) % 2).to_numpy()
rows, best_pv = [], pd.Series(V.pv.to_numpy(), index=V.index)
for line, df in V.groupby("line"):
    curve = {a: D2(df.act, tilt(df, a), df.w) for a in alphas}
    a_all = max(curve, key=curve.get)
    a_h = []
    for h in (0, 1):
        dh = df[half[df.index] == h]
        ch = {a: D2(dh.act, tilt(dh, a), dh.w) for a in alphas}
        a_h.append(max(ch, key=ch.get))
    best_pv.loc[df.index] = tilt(df, a_all).to_numpy()
    rows.append({"line": line, "D2 at 1": curve[1.0], "best alpha": a_all, "D2 at best": curve[a_all],
                 "gain": curve[a_all] - curve[1.0], "alpha half 0": a_h[0], "alpha half 1": a_h[1]})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print(f"\npooled D2: as is {D2(V.act, V.pv, V.w):.4f} | each line at its best alpha {D2(V.act, best_pv, V.w):.4f}")

                         rows   exposure  loss_rows    A/E
line      new_business                                    
buildings False         12588  10492.881        479  0.934
          True           2279   1000.763         35  0.688
contents  False         15863  13146.471        423  0.900
          True           2869   1281.892         64  1.352
motor     False         21688  14804.291       2716  1.086
          True           3915   1359.707        348  0.999 

     line  D2 at 1  best alpha  D2 at best   gain  alpha half 0  alpha half 1
buildings   0.0684        1.15      0.0698 0.0014          1.25          1.05
 contents   0.0539        1.10      0.0544 0.0005          1.00          1.20
    motor   0.0373        0.90      0.0380 0.0007          0.90          0.90

pooled D2: as is 0.1077 | each line at its best alpha 0.1086


In [3]:
rng = np.random.default_rng(2026)
B, CH = 2000, 50

def boot_ae(df, key):
    pol = df.groupby(key + ["policy_id"], observed=True).agg(A=("act", "sum"), E=("pv", "sum")).reset_index()
    line_ae = df.groupby("line").act.sum() / df.groupby("line").pv.sum()
    out = []
    for k, gp in pol.groupby(key, observed=True):
        k = k if isinstance(k, tuple) else (k,)
        A, E = gp.A.to_numpy(), gp.E.to_numpy()
        reps = np.concatenate([(c @ A) / (c @ E) for c in
                               (rng.poisson(1.0, size=(CH, len(A))) for _ in range(B // CH))])
        ae, se = A.sum() / E.sum(), reps.std()
        lo, hi = np.percentile(reps, [2.5, 97.5])
        row = dict(zip(key, k))
        row.update({"policies": len(A), "loss_policies": int((A > 0).sum()), "A/E": ae,
                    "95% low": lo, "95% high": hi, "z vs 1": (ae - 1) / se,
                    "z vs line": (ae - line_ae[row["line"]]) / se})
        out.append(row)
    return pd.DataFrame(out)

# exposure-weighted predicted deciles within each line
V = V.sort_values(["line", "rate"]).copy()
V["decile"] = (V.groupby("line").w.cumsum() / V.groupby("line").w.transform("sum") * 10).clip(upper=9.999).astype(int) + 1

print("=== line x new business ===")
print(boot_ae(V, ["line", "new_business"]).round(3).to_string(index=False))
print("\n=== line x predicted decile ===")
print(boot_ae(V, ["line", "decile"]).round(3).to_string(index=False))

=== line x new business ===
     line  new_business  policies  loss_policies   A/E  95% low  95% high  z vs 1  z vs line
buildings         False     12107            479 0.934    0.706     1.213  -0.506      0.175
buildings          True      2192             35 0.688    0.351     1.070  -1.703     -1.217
 contents         False     14982            423 0.900    0.735     1.090  -1.083     -0.441
 contents          True      2707             64 1.352    0.818     1.979   1.188      1.387
    motor         False     20084           2701 1.086    1.001     1.183   1.851      0.208
    motor          True      3638            346 0.999    0.823     1.208  -0.005     -0.774

=== line x predicted decile ===
     line  decile  policies  loss_policies   A/E  95% low  95% high  z vs 1  z vs line
buildings       1      1375             35 0.365    0.202     0.563  -6.775     -5.826
buildings       2      1413             44 0.855    0.430     1.390  -0.592     -0.227
buildings       3      1432

In [4]:
FIRE = 97.05
b = V[V.line == "buildings"].copy()
b["fire_share"] = FIRE / b.rate
tab = (b.groupby("decile")
         .apply(lambda g: pd.Series({"exposure": g.w.sum(),
                                     "mean rate": np.average(g.rate, weights=g.w),
                                     "fire share": np.average(g.fire_share, weights=g.w),
                                     "A/E": g.act.sum() / g.pv.sum()}))
         .round(3))
print(tab.to_string())
print(f"\nmin rate {b.rate.min():.2f} | rows with rate < 1.2 x fire: {(b.rate < 1.2 * FIRE).mean():.1%}")

        exposure  mean rate  fire share    A/E
decile                                        
1       1149.163    222.897       0.436  0.365
2       1149.493    241.022       0.403  0.855
3       1148.953    252.749       0.384  1.086
4       1149.539    263.176       0.369  0.814
5       1149.252    274.114       0.354  1.157
6       1149.731    287.058       0.338  1.184
7       1149.319    303.018       0.320  0.930
8       1149.422    327.736       0.296  0.588
9       1148.542    391.341       0.250  0.948
10      1150.230   1103.455       0.115  0.962

min rate 187.01 | rows with rate < 1.2 x fire: 0.0%


In [5]:
db = P.build_line("buildings", ours, claims)
X = V[V.line == "buildings"][["row_id", "decile", "w"]].merge(db["p"], on="row_id", how="left")
d1, mid = (X.decile == 1), X.decile.between(2, 9)

cat_rows, num_rows = [], []
for c in db["p"].columns:
    if c in ("row_id", "policy_id"):
        continue
    s = X[c]
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 20:
        r = s.rank(pct=True).fillna(0.5)
        num_rows.append({"feature": c, "d1 mean pct rank": np.average(r[d1], weights=X.w[d1]),
                         "d1 median": s[d1].median(), "d2-9 median": s[mid].median()})
    elif s.nunique() <= 50:
        s = s.astype(str)
        p1 = X.w[d1].groupby(s[d1]).sum() / X.w[d1].sum()
        pm = X.w[mid].groupby(s[mid]).sum() / X.w[mid].sum()
        diff = p1.sub(pm, fill_value=0)
        top = diff.idxmax()
        cat_rows.append({"feature": c, "TVD": 0.5 * diff.abs().sum(),
                         "most over-represented in d1": f"{top} ({p1.get(top, 0):.0%} vs {pm.get(top, 0):.0%})"})

print("=== categorical ===")
print(pd.DataFrame(cat_rows).sort_values("TVD", ascending=False).head(12).round(3).to_string(index=False))
print("\n=== numeric ===")
print(pd.DataFrame(num_rows).assign(dev=lambda x: (x["d1 mean pct rank"] - 0.5).abs())
        .sort_values("dev", ascending=False).head(12).round(3).to_string(index=False))

20:23:16 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
20:23:16 buildings: fire priced separately at 97.05 per exposure (developed)
20:23:17 buildings: own GLM for ['flood']; main GLM on the other perils
=== categorical ===
                  feature   TVD most over-represented in d1
  accidental_damage_cover 0.198          False (78% vs 58%)
           payment_method 0.165         annual (70% vs 54%)
         voluntary_excess 0.152              0 (59% vs 49%)
    policyholder_age_band 0.146          35-49 (23% vs 15%)
            renewal_count 0.118                3 (9% vs 5%)
             smoke_alarms 0.106           True (88% vs 74%)
             new_business 0.080          False (98% vs 90%)
                occupancy 0.047 owner_occupied (94% vs 89%)
                  channel 0.047  comparison_site (8% vs 5%)
      missed_payments_12m 0.045              0 (96% vs 91%)
                 mortgage 0.044           True (59% vs 55%)
customer_service_contacts 0.0

In [6]:
b = V[V.line == "buildings"]

# (a) frequency / severity by decile
pol = b.groupby(["decile", "policy_id"]).agg(A=("act", "sum"), E=("pv", "sum"), w=("w", "sum")).reset_index()
def summ(g):
    L = g.A[g.A > 0]
    return pd.Series({"exposure": g.w.sum(), "loss pol": len(L), "freq": len(L) / g.w.sum(),
                      "mean sev": L.mean(), "median sev": L.median(), "max loss": L.max(),
                      "max share": L.max() / L.sum(), "A/E": g.A.sum() / g.E.sum(),
                      "A/E w/o max": (L.sum() - L.max()) / g.E.sum()})
print("=== (a) frequency / severity by decile ===")
print(pol.groupby("decile").apply(summ).round(3).to_string())

# (b) accidental damage cover
print("\naccidental_damage_cover in model features:", "accidental_damage_cover" in db["feats"])
X2 = X.merge(b[["row_id", "act", "pv"]], on="row_id")
t = (X2.assign(group=np.where(X2.decile == 1, "d1", "d2-10"))
       .groupby(["accidental_damage_cover", "group"])
       .agg(exposure=("w", "sum"), loss_rows=("act", lambda s: int((s > 0).sum())), A=("act", "sum"), E=("pv", "sum")))
t["A/E"] = t.A / t.E
print("\n=== (b) A/E by accidental damage cover ===")
print(t.drop(columns=["A", "E"]).round(3).to_string())

=== (a) frequency / severity by decile ===
        exposure  loss pol   freq   mean sev  median sev    max loss  max share    A/E  A/E w/o max
decile                                                                                             
1       1149.163      35.0  0.030   2669.264    1636.416   14538.215      0.156  0.365        0.308
2       1149.493      44.0  0.038   5386.785    2985.157   45243.878      0.191  0.855        0.692
3       1148.953      42.0  0.037   7505.970    3960.527   59682.480      0.189  1.086        0.880
4       1149.539      57.0  0.050   4319.927    2523.397   37705.570      0.153  0.814        0.689
5       1149.252      54.0  0.047   6751.215    2024.649  167679.960      0.460  1.157        0.625
6       1149.731      39.0  0.034  10020.627    1499.364  300000.000      0.768  1.184        0.275
7       1149.319      56.0  0.049   5781.679    2173.602   66300.680      0.205  0.930        0.739
8       1149.422      56.0  0.049   3952.448    2052.481 

In [7]:
from pathlib import Path
for d in ["results/checks_part1", "results/checks_part2"]:
    p = Path(d)
    print(d, "exists" if p.exists() else "MISSING", sorted(x.name for x in p.glob("*")) if p.exists() else "")

results/checks_part1 exists ['leakage.csv', 'report.md', 'schema.json', 'step_scan_claims.csv', 'step_scan_policies.csv']
results/checks_part2 exists ['leakage.csv', 'report.md', 'schema.json', 'step_scan_claims.csv', 'step_scan_policies.csv']


In [8]:
X3 = X2.assign(line="buildings", adc=X2.accidental_damage_cover.astype(str))
print(boot_ae(X3, ["line", "adc"]).round(3).to_string(index=False))

     line   adc  policies  loss_policies   A/E  95% low  95% high  z vs 1  z vs line
buildings False      8576            292 0.770    0.586     0.981  -2.251     -1.378
buildings  True      5723            222 1.112    0.692     1.599   0.456      0.820


In [9]:
import datetime as dt
for f in sorted(Path("results/checks_part2").glob("*")):
    print(f"{f.name:28s} {dt.datetime.fromtimestamp(f.stat().st_mtime):%Y-%m-%d %H:%M}")

print("\n" + "=" * 30 + " report.md " + "=" * 30)
print(Path("results/checks_part2/report.md").read_text(encoding="utf-8"))

lk = pd.read_csv("results/checks_part2/leakage.csv")
print("\n" + "=" * 30 + " leakage.csv " + "=" * 30)
print(lk.columns.tolist())
print(lk.head(20).to_string(index=False))
print("\nlicence_points rows:")
print(lk[lk.apply(lambda r: r.astype(str).str.contains("licence", case=False).any(), axis=1)].to_string(index=False))

leakage.csv                  2026-10-23 14:32
report.md                    2026-10-23 14:32
schema.json                  2026-10-23 14:32
step_scan_claims.csv         2026-10-23 14:32
step_scan_policies.csv       2026-10-23 14:32

============================== report.md ==============================
# Data checks: data_part2/emerging

## 1. Schema differences from the reference

```
                         table          change                       detail
0            customers.parquet  missing column                  credit_band
1            customers.parquet  missing column                date_of_birth
2            customers.parquet      new column                 gender (str)
3             expenses.parquet       new table                             
4   policies_buildings.parquet  missing column             policyholder_age
5   policies_buildings.parquet      new column  policyholder_age_band (str)
6    policies_contents.parquet  missing column             policyholder_age
7   

In [10]:
for line in PART2["lines"]:
    dl = P.build_line(line, ours, claims)
    used = "customer_service_contacts" in dl["feats"]
    tr = dl["tr"] if "customer_service_contacts" in dl["tr"] else \
         dl["tr"].merge(dl["p"][["row_id", "customer_service_contacts"]], on="row_id", how="left")
    has = tr.target > 0
    print(f"\n{line}: used as feature = {used} | drop = {PART2.get('drop', {}).get(line)}")
    print(f"  mean contacts, train: with loss {tr.customer_service_contacts[has].mean():.3f}, "
          f"without loss {tr.customer_service_contacts[~has].mean():.3f} | test {dl['pt'].customer_service_contacts.mean():.3f}")


motor: used as feature = False | drop = ['previous_insurer', 'customer_tenure', 'soil_type', 'roof_type', 'energy_rating', 'building_age', 'property_type', 'bedrooms', 'flood_zone', 'elevation_m', 'distance_to_river_km', 'discount_applied', 'occupation']
  mean contacts, train: with loss 3.405, without loss 0.424 | test 0.400
20:26:46 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
20:26:46 buildings: fire priced separately at 97.05 per exposure (developed)
20:26:47 buildings: own GLM for ['flood']; main GLM on the other perils

buildings: used as feature = False | drop = ['previous_insurer', 'occupation', 'region', 'customer_tenure', 'discount_applied']
  mean contacts, train: with loss 3.111, without loss 0.448 | test 0.398
20:26:47 contents: sum_insured x1000 entries divided by 1000: train 57, test 135

contents: used as feature = False | drop = ['previous_insurer', 'occupation', 'customer_tenure', 'discount_applied']
  mean contacts, train: with loss 2.949

In [11]:
dm = P.build_line("motor", ours, claims)
print("licence_points in policies:", "licence_points" in dm["p"], "| in test:", "licence_points" in dm["pt"],
      "| in feats:", "licence_points" in dm["feats"])
if "licence_points" in dm["p"]:
    trm = dm["tr"] if "licence_points" in dm["tr"] else dm["tr"].merge(dm["p"][["row_id", "licence_points"]], on="row_id", how="left")
    has = trm.target > 0
    print(f"  dtype {dm['p'].licence_points.dtype} | mean, train: with loss {pd.to_numeric(trm.licence_points[has], errors='coerce').mean():.3f}, "
          f"without loss {pd.to_numeric(trm.licence_points[~has], errors='coerce').mean():.3f} | test {pd.to_numeric(dm['pt'].licence_points, errors='coerce').mean():.3f}")

print("\nLARGE_LOSS_CAP:", getattr(T, "LARGE_LOSS_CAP", None))
num = claims.select_dtypes("number")
print("raw claim columns with values == 300000:", {c: int((num[c] == 300000).sum()) for c in num.columns if (num[c] == 300000).any()})
print("validation rows with target == 300000:", V.groupby("line").act.apply(lambda s: int(np.isclose(s, 300000).sum())).to_dict())

licence_points in policies: False | in test: False | in feats: True

LARGE_LOSS_CAP: 500000
raw claim columns with values == 300000: {}
validation rows with target == 300000: {'buildings': 1, 'contents': 0, 'motor': 0}


In [12]:
# (a) where licence_points lives
for k, v in dm.items():
    if isinstance(v, pd.DataFrame):
        print(f"{k:>6}: rows {len(v):>7,} | licence_points: {'licence_points' in v}")
cu = pd.read_parquet(Path(PART2["data_dir"]) / "customers.parquet")
print("customers.parquet columns:", list(cu.columns))

# (b) loss vs no loss by year, vs test
trm = dm["tr"]
lp = pd.to_numeric(trm.licence_points, errors="coerce")
tab = trm.assign(lp=lp, loss=trm.target > 0).groupby(["year", "loss"]).lp.agg(["size", "mean"])
print("\n", tab.round(3).to_string())
for k, v in dm.items():
    if isinstance(v, pd.DataFrame) and "licence_points" in v and len(v) > 80000:
        print(f"test-like frame '{k}' mean licence_points: {pd.to_numeric(v.licence_points, errors='coerce').mean():.3f}")

# (c) the 300,000 row
r = V[(V.line == "buildings") & np.isclose(V.act, 300000)][["row_id", "policy_id", "w", "act", "pv"]]
print("\n", r.merge(db["p"][["row_id", "sum_insured"]], on="row_id").to_string(index=False))
pid = r.policy_id.iloc[0]
pc = [c for c in claims.columns if "policy" in c.lower()][0]
print(claims[claims[pc].astype(str) == pid].to_string(index=False))

     p: rows  47,283 | licence_points: False
    pt: rows  90,521 | licence_points: False
    tr: rows  47,283 | licence_points: True
    te: rows  90,521 | licence_points: True
customers.parquet columns: ['customer_id', 'surname', 'gender', 'occupation', 'marital_status', 'household_size', 'prior_claims_5y', 'licence_points', 'years_at_address', 'customer_since_year', 'email_domain', 'marketing_consent', 'document_preference', 'previous_insurer', 'campaign_code', 'branch_id', 'marketing_segment']

              size   mean
year loss               
2023 False  18653  0.902
     True    3027  0.939
2024 False  22539  0.899
     True    3064  0.914
test-like frame 'te' mean licence_points: 0.903

    row_id policy_id        w      act        pv  sum_insured
B00081492 B80127511 0.950267 300000.0 272.87146     362000.0
  claim_id policy_id      line peril  loss_date report_date status  paid_to_date  incurred_to_date  recoveries repairer_id third_party_id  report_lag  loss_year
CL86427090 B

In [13]:
for line, dl in [("motor", dm), ("buildings", db), ("contents", P.build_line("contents", ours, claims))]:
    if "prior_claims_5y" not in dl["tr"]:
        print(f"{line}: prior_claims_5y not in training table"); continue
    x = pd.to_numeric(dl["tr"].prior_claims_5y, errors="coerce")
    t = dl["tr"].assign(x=x, loss=dl["tr"].target > 0).groupby(["year", "loss"]).x.mean().unstack()
    te = pd.to_numeric(dl["te"].prior_claims_5y, errors="coerce").mean() if "prior_claims_5y" in dl["te"] else float("nan")
    print(f"\n{line}: in feats = {'prior_claims_5y' in dl['feats']} | test mean {te:.3f}")
    print(t.round(3).to_string())

import re
cfg_src = Path("src/config.py").read_text(encoding="utf-8").splitlines()
print("\nconfig lines with 300000 / 300_000 / cap:")
for i, l in enumerate(cfg_src, 1):
    if re.search(r"300_?000|\bcap\b", l, re.I):
        print(f"{i:4d}: {l}")

20:28:51 contents: sum_insured x1000 entries divided by 1000: train 57, test 135

motor: in feats = True | test mean 0.264
loss  False  True 
year              
2023  0.258  0.327
2024  0.258  0.324

buildings: in feats = True | test mean 0.265
loss  False  True 
year              
2023  0.267  0.318
2024  0.267  0.317

contents: in feats = True | test mean 0.268
loss  False  True 
year              
2023  0.272  0.315
2024  0.272  0.361

config lines with 300000 / 300_000 / cap:


In [14]:
import re
print("cap definitions in src:")
for f in sorted(Path("src").glob("*.py")):
    for i, l in enumerate(f.read_text(encoding="utf-8").splitlines(), 1):
        if re.search(r"300_?000|3e5|300000\.?0?\b|CAP\b|\"cap\"|'cap'", l):
            print(f"  {f.name}:{i}: {l.strip()}")

dec = """### 2026-10-24 20:30 — Redistribution diagnostics on the 2024 validation; Part 2 data checks reviewed (notebook 27)
- Rows: results/valid2024_rows_ours.csv (59,202 rows; pooled D2 0.1077, same as the E9-P2b rerun). Not committed.
- Slope alpha (rate x (rate/mean)^alpha, total held), chosen per line on 2024 and on two policy halves: buildings 1.15 (halves 1.25 / 1.05), contents 1.10 (1.00 / 1.20), motor 0.90 (0.90 / 0.90); pooled gain +0.0009 in-sample. Rule set before the run (both halves on one side of 1 and pooled gain >= +0.002) not met -> no tilt.
- New business by line, policy bootstrap (2,000 reps): buildings A/E 0.69 [0.35, 1.07], contents 1.35 [0.82, 1.98], motor 1.00 [0.82, 1.21]. All include 1 -> one NB load for all lines kept.
- Buildings lowest decile A/E 0.365 [0.20, 0.56] is large-loss volatility, not mispricing: largest loss in decile 1 is 14.5k; A/E without each decile's largest loss ranges 0.28-0.88 (decile 6: one 300,000 loss is 77% of its losses). The policy bootstrap cannot produce unseen large losses, so buildings decile intervals are too narrow. A flat fire rate acting as a floor was checked and rejected (fire share 0.44 vs 0.40 in deciles 1 and 2).
- Buildings accidental_damage_cover: A/E without cover 0.77 [0.59, 0.98], with cover 1.11 [0.69, 1.60]; intervals overlap; vs line A/E 0.91, z = -1.4 -> not adjusted (Limitations).
- Conclusion with the E9-P2b rerun: the 2024 validation does not support finer redistribution than the current model.
- Data checks (results/checks_part2, run Oct 23) reviewed: customer_service_contacts is post-claim (mean 2.9-3.4 with a loss vs 0.42-0.45 without; test 0.40 on every line), confirmed leakage, not a model feature on any line. licence_points (customers.parquet snapshot): 0.91-0.94 with a loss vs 0.90 without, test 0.903 -> no time leakage; backlog item closed. prior_claims_5y: 0.32-0.36 with a loss vs 0.26-0.27 without, test 0.26-0.27; the loss-row mean is far below 1, so a policy's own claims are not counted -> no leakage. Targets of exactly 300,000 are the buildings cap applied after adjustment (fire claim, incurred 361,093, sum insured 362,000), not a raw value.
"""
p = Path("DECISIONS.md"); t = p.read_text(encoding="utf-8")
assert "Redistribution diagnostics on the 2024 validation" not in t, "already recorded"
anchor = "## Assumptions register"
assert t.count(anchor) == 1
p.write_text(t.replace(anchor, dec + "\n" + anchor), encoding="utf-8")

log = ("| 111 | H | AI read the buildings lowest-decile A/E (0.365, bootstrap interval excluding 1) as a real 2.7x overcharge and proposed a flat fire rate as the cause. "
       "The team asked whether frequency and severity had been checked. "
       "| Fire-share table rejected the floor; the frequency / severity split by decile showed decile 1 simply had no large loss (A/E without the largest loss 0.28-0.88 across deciles). "
       "| Not adjusted; recorded as large-loss volatility (DECISIONS, notebook 27). Lesson: on a lumpy line, split A/E into frequency and severity and remove the largest loss before reading a segment as mispriced; a policy bootstrap cannot produce unseen large losses. |\n")
q = Path("AI_USE_LOG.md"); lines = q.read_text(encoding="utf-8").splitlines(keepends=True)
assert not any(l.startswith("| 111 |") for l in lines), "111 already exists"
idx = [i for i, l in enumerate(lines) if l.startswith("| 110 |")]
assert len(idx) == 1
lines.insert(idx[0] + 1, log)
q.write_text("".join(lines), encoding="utf-8")
print("\nrecorded")

cap definitions in src:
  pipeline.py:269: used = n / tr.weight.sum() * expected_excess(spec["threshold"], adj["cap"], spec["alpha"])
  pipeline.py:273: fit = pareto_loading(mc.net_adj_uncapped, tr.weight.sum(), spec["threshold"], adj["cap"])
  target.py:21: LARGE_LOSS_CAP = 500_000
  target.py:84: def match_claims(claims, policies, line, excess_on_liability=True, cap=LARGE_LOSS_CAP, amounts_net=None):
  target.py:110: cap=None gives the uncapped target; pass LARGE_LOSS_CAP for the modelling target.
  target.py:188: "cap": 500_000,
  target.py:194: "cap": 300_000,   # just above the 99.9th percentile (same rule as motor); final level by validation
  target.py:200: "cap": None,
  target.py:223: cap = adj["cap"]
  target.py:233: a["cap"] = cap

recorded


In [2]:
import os, sys, re, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import src.pipeline as P, src.target as T
from src.config import PART2
from src.target import load_claims

ours = copy.deepcopy(PART2)
T.AMOUNTS_NET, T.ROW_COMPLETENESS = bool(ours["amounts_net"]), bool(ours["row_completeness"])
claims = load_claims(ours["data_dir"])
for line in ours["lines"]:
    dl = P.build_line(line, ours, claims)
    print(f"{line:9s} | in training table: {'gender' in dl['tr']} | in feats: {'gender' in dl['feats']} "
          f"| drop: {'gender' in ours.get('drop', {}).get(line, [])}")

print("\nDECISIONS lines mentioning gender / sex:")
for i, l in enumerate(Path("DECISIONS.md").read_text(encoding="utf-8").splitlines(), 1):
    if re.search(r"\bgender\b|\bsex\b|성별", l, re.I):
        print(f"{i:5d}: {l.strip()[:200]}")

motor     | in training table: True | in feats: True | drop: False
20:33:14 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
20:33:15 buildings: fire priced separately at 97.05 per exposure (developed)
20:33:15 buildings: own GLM for ['flood']; main GLM on the other perils
buildings | in training table: True | in feats: True | drop: False
20:33:15 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
contents  | in training table: True | in feats: True | drop: False

DECISIONS lines mentioning gender / sex:
  506: - Columns: age bands instead of ages; no credit_band, annual_mileage, telematics; **gender added** (allowed
  572: - **gender:** used (allowed for pricing in this market, Dataset Description).
  683: third_party_prop +0.0033 (ncb_years × years_licensed, driver_age_band × gender). All other perils negative.
  931: level factor for motor, so emerging-only effects (gender, regions E01-E06) are not learned. Tested as E21.


In [3]:
import numpy as np, pandas as pd
rows = []
for line in ours["lines"]:
    d = P.build_line(line, ours, claims)
    tr0 = d["tr"].merge(d["p"][["row_id", "policy_id"]], on="row_id")
    vy, tys = ours["folds"][0]
    tr, va = tr0[tr0.year.isin(tys)], tr0[tr0.year == vy]
    rM, _ = P.predict_rate(line, ours, tr, va.assign(gender="M"), d["feats"])
    rF, _ = P.predict_rate(line, ours, tr, va.assign(gender="F"), d["feats"])
    rM, rF = np.asarray(rM, float), np.asarray(rF, float)
    rows.append({"line": line, "training": ours.get("developed_training", {}).get(line, "emerging only"),
                 "share F": float((va.gender == "F").mean()),
                 "F/M mean prediction": np.average(rF, weights=va.weight) / np.average(rM, weights=va.weight),
                 "rows where F != M": float(np.mean(~np.isclose(rF, rM)))})
print(pd.DataFrame(rows).round(4).to_string(index=False))

20:33:56 motor: developed training table 979,968 rows
20:34:47 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
20:34:47 buildings: fire priced separately at 97.05 per exposure (developed)
20:34:47 buildings: own GLM for ['flood']; main GLM on the other perils
20:34:52 buildings: developed training table 707,726 rows
20:35:07 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
20:35:11 contents: developed training table 721,617 rows
     line      training  share F  F/M mean prediction  rows where F != M
    motor     developed   0.5054               1.0000                0.0
buildings emerging only   0.5000               0.9738                1.0
 contents      combined   0.5010               1.0000                0.0


In [4]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
import src.pipeline as P, src.target as T
from src.config import PART2
from src.evaluate import evaluate

base = copy.deepcopy(PART2)
T.AMOUNTS_NET, T.ROW_COMPLETENESS = True, True
claims = T.load_claims(base["data_dir"])
alt = copy.deepcopy(base); alt["premium_lines"] = list(base["lines"])

rows, pooled = [], {"base": [], "prem_rel": []}
for line in base["lines"]:
    for name, cfg in [("base", base), ("prem_rel", alt)]:
        d = P.build_line(line, cfg, claims)
        tr0 = d["tr"]; vy, tys = cfg["folds"][0]
        tr, va = tr0[tr0.year.isin(tys)], tr0[tr0.year == vy]
        rate, info = P.predict_rate(line, cfg, tr, va, d["feats"])
        pv = (np.asarray(rate) + d.get("sep_rate", 0.0)) * va.weight.to_numpy()
        act = (va.target_all if "target_all" in va else va.target).to_numpy()
        r = evaluate(pd.Series(act), pv, pd.Series(va.weight.to_numpy()), show=False)
        lf = r["lift"]["A/E"]
        rows.append({"line": line, "model": name, "D2": r["D2"], "low": lf.iloc[0], "top": lf.iloc[-1],
                     "prem_rel used": "prem_rel" in d["feats"]})
        pooled[name].append(pd.DataFrame({"a": act, "p": pv, "w": va.weight.to_numpy()}))
for name, parts in pooled.items():
    a = pd.concat(parts); rows.append({"line": "ALL", "model": name, "D2": evaluate(a.a, a.p, a.w, show=False)["D2"]})
print(pd.DataFrame(rows).round(4).to_string(index=False))

20:40:45 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
20:40:46 buildings: fire priced separately at 97.05 per exposure (developed)
20:40:46 buildings: own GLM for ['flood']; main GLM on the other perils
20:40:54 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
20:40:54 buildings: fire priced separately at 97.05 per exposure (developed)
20:40:55 buildings: own GLM for ['flood']; main GLM on the other perils
20:41:03 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
20:41:24 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
     line    model     D2    low    top prem_rel used
    motor     base 0.0373 0.7940 0.8936         False
    motor prem_rel 0.0371 0.8114 0.8781          True
buildings     base 0.0684 0.3477 0.9613         False
buildings prem_rel 0.0688 0.5362 0.9618          True
 contents     base 0.0539 0.7371 0.9923         False
 contents prem_rel 0.0553 0.8555 0.9723          Tru

In [5]:
import os, sys, py_compile; from pathlib import Path; os.chdir(Path.home()/"IAA")

c = Path("src/config.py"); t = c.read_text(encoding="utf-8")
anchor = 'PART2["row_completeness"] = True              # Oct 24 (E22): completeness of each row\'s own period\n'
assert t.count(anchor) == 1 and "E23" not in t
t = t.replace(anchor, anchor + 'PART2["premium_lines"] = ["contents"]         # Oct 24 (E23): relative premium, contents only (+0.0014 D2)\n')
c.write_text(t, encoding="utf-8"); py_compile.compile(str(c), doraise=True)

p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
old_g = "- **gender:** used (allowed for pricing in this market, Dataset Description)."
assert d.count(old_g) == 1, "gender line differs; fix by hand"
d = d.replace(old_g, "- **gender:** allowed for pricing in this market (Dataset Description) and in every feature list, but "
              "effective in buildings only (F/M 0.974; emerging-only training). Motor (developed relativities) and contents "
              "(combined) drop it because the developed table has no gender column (F/M 1.000, checked Oct 24 notebook 26); "
              "motor region levels E01-E06 are likewise not learned. Restoring them was tested as E21 (+0.0002 / +0.0004).")
dec = """### E23: relative premium on the v5 structure (Oct 24, notebook 27) — rule fixed before running
- Column audit of both data folders against the feature lists (src/features.py EXCLUDE, config DROP, predict_rate):
  every exclusion re-checked with a one-way fit 2023 -> 2024; only written_premium was excluded without a test on the current structure (E16 +0.0035 on the old base).
- Rule (Part 2 GLM rule): adopt per line if 2024 D2 +0.0005 or more and neither tail moves more than 0.1 further from 1.
- Result, base -> prem_rel: motor 0.0373 -> 0.0371 (no); buildings 0.0684 -> 0.0688 (+0.0004, no; low decile 0.35 -> 0.54);
  contents 0.0539 -> 0.0553 (+0.0014; low 0.74 -> 0.86, top 0.99 -> 0.97) -> **adopted in contents** (as in Part 1). Pooled all-lines +0.0003.
- Not the cause of the leaderboard gap. Limitation (as Part 1): circular, inherits the insurer's own relativities.
- **p2_v6 (v5 + E23) expected public, written before submitting: 0.1425-0.1450** (contents gain about half carried over, as before).
  v6 becomes the final unless its public score is below 0.1420 (then investigate, keep v5).
"""
a = "## Assumptions register"; assert d.count(a) == 1 and "### E23:" not in d
p.write_text(d.replace(a, dec + "\n" + a), encoding="utf-8")
print("config patched, DECISIONS recorded")

config patched, DECISIONS recorded
